In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import sys, os

sys.path.insert(0, os.path.abspath(".."))

from msvar import em_fit, build_regressors
from src.constants import COLOR_EXP as COLOR_0, COLOR_REC as COLOR_1, SHADE_COL, SHADE_A, LW
from src.data_loader import load_macro_data, load_asset_data
from src.preprocess import prepare_macro_data, train_test_split
from src.plots import (plot_regime_states, plot_regime_probs,
                        plot_series_colored, plot_series_shaded,
                        plot_frontiers, plot_regime_nber, _fmt_matrix)
from src.portfolio import (estimate_regime_moments, optimize_regimes,
                            efficient_frontier, shrinkage_diagnostics)

plot_msm_states      = lambda d, s, title="MSM Regime States": plot_regime_states(d, s, title)
plot_msm_state_probs = lambda d, p, title="MSM State Probabilities": plot_regime_probs(d, p, title)

## Description

This notebook implements regime detection using **MSMH-VAR(K=2, p=1)** — a Markov-Switching
Mean-Heteroskedastic Vector Autoregression. Conditional on state k, macro indicators follow
a VAR(1) process rather than being i.i.d. draws as in the HMM.

The Hamilton filter and Kim smoother are used for the E-step; weighted OLS for the M-step.

## Data Preparation

The VAR(p) model consumes p observations as lags, so effective sample size is T-p.

In [ ]:
macro_df = load_macro_data("../data/macro_clean.csv")

In [ ]:
macro_df.info()

In [ ]:
X_train, X_test, scaler, dates_train, dates_test = prepare_macro_data(macro_df)

## Model fitting

Fit MSMH-VAR(K=2, p=1) via EM with multiple random restarts.

In [ ]:
result = em_fit(
    data=X_train,
    K=2,
    p=1,
    n_restarts=5,
    max_iter=300,
    tol=1e-6,
    random_state=42,
    verbose=True
)

In [ ]:
result.summary()

## Results extraction

In [ ]:
smoothed_probs = result.xi_smoothed
print("Smoothed probs shape:", smoothed_probs.shape)
print("Sum to 1:", np.allclose(smoothed_probs.sum(axis=0), 1.0))

In [ ]:
smoothed_probs_df = pd.DataFrame(
    smoothed_probs.T,
    index=dates_train[1:],
    columns=[f"Regime {i}" for i in range(smoothed_probs.shape[0])]
)
smoothed_probs_df.plot(figsize=(12, 6), title="Smoothed Regime Probabilities")
plt.show()

In [ ]:
filtered_probs = result.xi_filtered
print("Filtered probs shape:", filtered_probs.shape)
print("Sum to 1:", np.allclose(filtered_probs.sum(axis=0), 1.0))

In [ ]:
states_sequence = result.regime_sequence(use_smoothed=True)
print("State counts:", np.unique(states_sequence, return_counts=True))

### Transition matrix

In [ ]:
trans_mat = result.P
pd.DataFrame(
    trans_mat,
    columns=[f"Regime {k}" for k in range(result.K)],
    index=[f"Regime {k}" for k in range(result.K)]
)

In [ ]:
print("Average duration in each state (expected number of consecutive months):")
for i, d in enumerate(result.expected_durations):
    print(f"  State {i}: {d:.2f} months")

## Visualization of results

In [ ]:
dates_smooth = dates_train[result.p:]

In [ ]:
plot_msm_states(dates_smooth, states_sequence, title="MSM Predicted States Over Time")
plt.show()

In [ ]:
macro_df_aligned = macro_df.loc[dates_smooth].copy()
macro_df_expanded = macro_df_aligned.copy()
macro_df_expanded["MSM State"] = states_sequence

In [ ]:
asset_df = load_asset_data("../data/market_clean.csv")
asset_df.columns = ['Equity', 'Bonds', 'Gold']

asset_df_aligned = asset_df.loc[dates_smooth].copy()
asset_df_expanded = asset_df_aligned.copy()
asset_df_expanded["MSM State"] = macro_df_expanded["MSM State"]

In [ ]:
plot_series_colored(macro_df_expanded, series_name='vix', state_col='MSM State')
plt.show()

In [ ]:
for col in asset_df_aligned.columns:
    plot_series_colored(asset_df_expanded, series_name=col, state_col="MSM State", ax=None)
    plt.show()

## Portfolio optimization

In [ ]:
market_train, market_test = train_test_split(asset_df_expanded)
print("Last train date :", market_train.index[-1])
print("First test date :", market_test.index[0])

In [ ]:
moments = estimate_regime_moments(
    market_train,
    asset_cols=market_train.columns[:-1].tolist(),
    state_col="MSM State"
)

In [ ]:
shrinkage_diagnostics(moments, asset_cols=market_train.columns[:-1].tolist())

In [ ]:
asset_names = market_train.columns[:-1]

print("=" * 55)
print("Regime-Conditional MVO")
print("=" * 55)
results = optimize_regimes(moments, asset_names, rf=0.0, long_only=False)

print("\n" + "=" * 55)
print("Efficient Frontiers")
print("=" * 55)
plot_frontiers(moments, asset_names, results)
plt.show()

### Target weights export

In [ ]:
output_path = os.path.join("..", "outputs/tables")
weights_by_regime = {
    k: results[k]['min_variance'].weights
    for k in results
}

states_sequence_series = pd.Series(states_sequence, index=dates_train[1:], name='MSVAR State')
# Switching labels: 0 -> 1, 1 -> 0
states_sequence_series += 1
states_sequence_series[states_sequence_series == 2] = 0

rows = []
for date, regime in states_sequence_series.items():
    w = weights_by_regime[regime]
    rows.append({
        'Date':   date,
        'Regime': regime,
        'Equity': w[0],
        'Bonds':  w[1],
        'Gold':   w[2],
    })

df_target_weights = pd.DataFrame(rows).set_index('Date')
df_target_weights.to_csv(os.path.join(output_path, "msvar_target_weights.csv"))
df_target_weights.head()

### MVO (Max-Sharpe / Tangency) Portfolio

In [ ]:
from src.portfolio import MeanVariancePortfolio

mvo_weights_by_regime = {}
print('MVO (Tangency) Weights:')
print('=' * 55)
for k, m in moments.items():
    mvp = MeanVariancePortfolio(
        mu=m['mu'], Sigma=m['sigma'],
        asset_names=list(asset_names),
        rf=0.0, long_only=True, regime=k,
    )
    res = mvp.tangency()
    mvo_weights_by_regime[k] = res.weights
    print(f'  Regime {k}: Equity={res.weights[0]:.3f}  '
          f'Bonds={res.weights[1]:.3f}  Gold={res.weights[2]:.3f}  '
          f'Sharpe(ann)={res.sharpe * (12**0.5):.3f}')

rows_mvo = []
for date, regime in states_sequence_series.items():
    w = mvo_weights_by_regime[regime]
    rows_mvo.append({'Date': date, 'Regime': regime,
                     'Equity': w[0], 'Bonds': w[1], 'Gold': w[2]})

df_mvo_weights = pd.DataFrame(rows_mvo).set_index('Date')
df_mvo_weights.to_csv(os.path.join(output_path, 'msvar_mvo_target_weights.csv'))
print('\nSaved: msvar_mvo_target_weights.csv')
df_mvo_weights.head()

In [ ]:
from src.backtest import backtest_with_transaction_costs
from src.data_loader import load_asset_data
from src.constants import ASSET_COLS

asset_returns_full = load_asset_data('../data/market_clean.csv')[ASSET_COLS].sort_index()

mvo_test_weights = df_mvo_weights.rename(columns={
    'Equity': 'index_fund', 'Bonds': 'treasury_fund', 'Gold': 'gold_fund'
})

SIGNAL_START_MSVAR = pd.Timestamp('2018-11-30')
mvo_test_weights = mvo_test_weights.loc[mvo_test_weights.index >= SIGNAL_START_MSVAR].copy()

result_mvo = backtest_with_transaction_costs(
    target_weights=mvo_test_weights,
    asset_returns=asset_returns_full,
    transaction_cost_bps=10,
    apply_signal_lag=True,
    include_initial_trade_cost=False,
)

first_date = result_mvo.index[0]
result_mvo.loc[first_date, 'trade_notional']         = 0.0
result_mvo.loc[first_date, 'one_way_turnover']       = 0.0
result_mvo.loc[first_date, 'transaction_cost']       = 0.0
result_mvo.loc[first_date, 'net_return_after_costs'] = result_mvo.loc[first_date, 'gross_return']

result_mvo.to_csv(os.path.join(output_path, 'msvar_mvo_monthly_returns_with_costs.csv'))
print(f'Test period: {result_mvo.index.min().date()} to {result_mvo.index.max().date()}')

net_ret = result_mvo['net_return_after_costs']
geo_mean = (1 + net_ret).prod() ** (12 / len(net_ret)) - 1
vol = net_ret.std() * np.sqrt(12)
sharpe = (geo_mean - 0.02664) / vol
print(f'\nMSVAR-MVO (net of costs):')
print(f'  Ann. return (geo): {geo_mean*100:.2f}%')
print(f'  Ann. volatility  : {vol*100:.2f}%')
print(f'  Sharpe ratio     : {sharpe:.3f}')

## Regime-Shaded Indicator Plots: VIX, FEDFUNDS, and UNRATE

Line color reflects model-classified state; NBER recession bands provide an independent reference.
Note: recession_state=0 for MSVAR (label 0 = contraction).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='vix', state_col='MSM State',
    recession_state=0,
    title='CBOE VIX: Model Regime vs NBER Recessions',
    ylabel='VIX (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='fed_funds_rate', state_col='MSM State',
    recession_state=0,
    title='Effective Federal Funds Rate: Model Regime vs NBER Recessions',
    ylabel='FEDFUNDS (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
plot_regime_nber(
    macro_df_expanded, series_name='unemployment_rate', state_col='MSM State',
    recession_state=0,
    title='Civilian Unemployment Rate: Model Regime vs NBER Recessions',
    ylabel='UNRATE (standardized)', ax=ax
)
plt.tight_layout()
plt.show()

## Robustness Check: K = 3 Regimes

AIC and BIC are used for model selection (LR test is non-standard under H0: K=2).

In [ ]:
result_k3 = em_fit(
    data=X_train,
    K=3,
    p=1,
    n_restarts=3,
    max_iter=200,
    tol=1e-5,
    random_state=42,
    verbose=True
)

In [ ]:
T = X_train.shape[0]
ll_k2 = result.log_likelihood
n_params_k2 = result.n_params
ll_k3 = result_k3.log_likelihood
n_params_k3 = result_k3.n_params

lr_stat = 2 * (ll_k3 - ll_k2)
delta_df = n_params_k3 - n_params_k2

summary_k3 = pd.DataFrame({
    'K': [2, 3],
    'Log-Likelihood': [ll_k2, ll_k3],
    'N Parameters':   [n_params_k2, n_params_k3],
    'AIC': [-2*ll_k2 + 2*n_params_k2, -2*ll_k3 + 2*n_params_k3],
    'BIC': [-2*ll_k2 + n_params_k2*np.log(T), -2*ll_k3 + n_params_k3*np.log(T)],
})
print(f'LR statistic (K=2 vs K=3): {lr_stat:.2f}  (delta_df = {delta_df})')
summary_k3

In [ ]:
states_k3 = result_k3.regime_sequence(use_smoothed=True)
print('K=3 state counts:', dict(zip(*np.unique(states_k3, return_counts=True))))

key_vars = ['vix', 'credit_spread', 'unemployment_rate',
            'industrial_production', 'fed_funds_rate']
key_idx  = [list(macro_df.columns).index(v) for v in key_vars]

means_k3 = pd.DataFrame(
    {f'State {k}': result_k3.B_list[k][:, 0][key_idx] for k in range(3)},
    index=key_vars
)
means_k3

In [ ]:
summary_k3.to_csv(os.path.join(output_path, 'msvar_k3_model_selection.csv'), index=False)
means_k3.to_csv(os.path.join(output_path, 'msvar_k3_regime_means.csv'))
print('Saved K=3 MSVAR results.')